# 03 — Case Retrieval
**Tujuan:** Bangun sistem pencarian kasus mirip dengan dua pendekatan:
- **TF-IDF + SVM / Naive Bayes**
- **Sentence-BERT** (`paraphrase-multilingual-MiniLM`)

```
projek-cbr-hukum/
└── data/
    ├── processed/   ← input  (cases.csv)
    └── eval/        ← output (queries.json, train/test idx)
```

In [2]:
import re, sys, json, pickle, warnings
from pathlib import Path
from typing import List, Tuple
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
warnings.filterwarnings('ignore')

BASE          = Path('..').resolve()
PROCESSED_DIR = BASE / 'data' / 'processed'
EVAL_DIR      = BASE / 'data' / 'eval'
MODELS_DIR    = BASE / 'models'
for d in [EVAL_DIR, MODELS_DIR]: d.mkdir(parents=True, exist_ok=True)

# ── Load data ────────────────────────────────────────────────
df = pd.read_csv(PROCESSED_DIR / 'cases.csv')
df['text_combined'] = (
    df['ringkasan_fakta'].fillna('') + ' ' +
    df['pasal'].fillna('')            + ' ' +
    df['amar_putusan'].fillna('')
)
print(f'Dataset: {len(df)} kasus | Label unik: {df["label_vonis"].unique()}')

Dataset: 60 kasus | Label unik: <ArrowStringArray>
['tidak_diketahui', 'bebas']
Length: 2, dtype: str


In [3]:
# ════════════════════════════════════════════════════════════
# PREPROCESSING TEKS
# ════════════════════════════════════════════════════════════
STOPWORDS = {
    'yang','dan','di','ke','dari','dengan','untuk','pada','dalam',
    'oleh','ini','itu','telah','tidak','ada','adalah','atau','juga',
    'bahwa','tersebut','hal','nomor','tanggal','tahun','kepada',
    'sebagai','tentang','atas','akan','serta','antara','menurut',
    'sebagaimana','catatan','pengadilan','negeri','mahkamah','agung',
}

def preprocess(text: str) -> str:
    text = text.lower()
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return ' '.join(t for t in text.split() if t not in STOPWORDS and len(t) > 2)

# Terapkan ke semua dokumen
df['text_preprocessed'] = [preprocess(t) for t in tqdm(df['text_combined'], desc='Preprocessing')]
print('Preprocessing selesai ')

Preprocessing: 100%|██████████| 60/60 [00:00<00:00, 2697.36it/s]

Preprocessing selesai 


In [4]:
# ════════════════════════════════════════════════════════════
# TRAIN / TEST SPLIT  (80 : 20)
# ════════════════════════════════════════════════════════════

# Filter label yang ada minimal 2 dokumen (untuk stratify)
valid_labels = df['label_vonis'].value_counts()
valid_labels = valid_labels[valid_labels >= 2].index.tolist()
df_f = df[df['label_vonis'].isin(valid_labels)].copy().reset_index(drop=True)

le = LabelEncoder()
y  = le.fit_transform(df_f['label_vonis'])

idx_all = list(range(len(df_f)))
idx_train, idx_test, y_train, y_test = train_test_split(
    idx_all, y, test_size=0.2, random_state=42, stratify=y
)

X_all   = df_f['text_preprocessed'].tolist()
X_train = [X_all[i] for i in idx_train]
X_test  = [X_all[i] for i in idx_test]

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Label: {list(le.classes_)}')

# Simpan index
pd.Series(idx_train).to_csv(EVAL_DIR / 'train_idx.csv', index=False, header=['idx'])
pd.Series(idx_test).to_csv( EVAL_DIR / 'test_idx.csv',  index=False, header=['idx'])

Train: 48 | Test: 12
Label: ['bebas', 'tidak_diketahui']


In [5]:
# ════════════════════════════════════════════════════════════
# PENDEKATAN A — TF-IDF  +  SVM  +  NAIVE BAYES
# ════════════════════════════════════════════════════════════

tfidf = TfidfVectorizer(
    max_features = 5000,
    ngram_range  = (1, 2),
    min_df       = 1,
    max_df       = 0.95,
    sublinear_tf = True,
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
X_all_tfidf   = tfidf.transform(X_all)

# ── SVM ─────────────────────────────────────────────────────
svm = LinearSVC(C=1.0, max_iter=5000, random_state=42)
svm.fit(X_train_tfidf, y_train)
svm_pred = svm.predict(X_test_tfidf)

print('=== SVM ===')
print(classification_report(le.inverse_transform(y_test),
                             le.inverse_transform(svm_pred), zero_division=0))

# ── Naive Bayes ─────────────────────────────────────────────
nb = MultinomialNB(alpha=0.5)
nb.fit(X_train_tfidf.toarray(), y_train)
nb_pred = nb.predict(X_test_tfidf.toarray())

print('=== Naive Bayes ===')
print(classification_report(le.inverse_transform(y_test),
                            le.inverse_transform(nb_pred), zero_division=0))

# ── Simpan model ────────────────────────────────────────────
import pickle
pickle.dump(tfidf, open(MODELS_DIR / 'tfidf_vectorizer.pkl', 'wb'))
pickle.dump(svm,   open(MODELS_DIR / 'svm_model.pkl',        'wb'))
pickle.dump(nb,    open(MODELS_DIR / 'nb_model.pkl',          'wb'))
pickle.dump(le,    open(MODELS_DIR / 'label_encoder.pkl',     'wb'))
print('Model TF-IDF + SVM + NB tersimpan')

=== SVM ===
                 precision    recall  f1-score   support

          bebas       0.00      0.00      0.00         3
tidak_diketahui       0.73      0.89      0.80         9

       accuracy                           0.67        12
      macro avg       0.36      0.44      0.40        12
   weighted avg       0.55      0.67      0.60        12

=== Naive Bayes ===
                 precision    recall  f1-score   support

          bebas       0.33      0.33      0.33         3
tidak_diketahui       0.78      0.78      0.78         9

       accuracy                           0.67        12
      macro avg       0.56      0.56      0.56        12
   weighted avg       0.67      0.67      0.67        12

Model TF-IDF + SVM + NB tersimpan


In [6]:
# ════════════════════════════════════════════════════════════
# PENDEKATAN B — SENTENCE-BERT
# ════════════════════════════════════════════════════════════
# Lebih ringan dari IndoBERT mentah — bisa jalan di CPU
# Set False jika RAM < 4 GB
BERT_ENABLED = True

bert_model      = None
bert_embeddings = None

if BERT_ENABLED:
    from sentence_transformers import SentenceTransformer
    BERT_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
    print(f'Memuat {BERT_NAME} ...')
    bert_model = SentenceTransformer(BERT_NAME)

    texts_bert = [
        (str(row['ringkasan_fakta']) + ' ' + str(row['pasal']))[:512]
        for _, row in df_f.iterrows()
    ]
    print('Menghitung embeddings...')
    bert_embeddings = bert_model.encode(
        texts_bert, batch_size=16,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    np.save(str(MODELS_DIR / 'bert_embeddings.npy'), bert_embeddings)
    print(f'Embeddings shape: {bert_embeddings.shape} ')
else:
    print('BERT dinonaktifkan — hanya TF-IDF yang digunakan.')

Memuat sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 372.91it/s]


Menghitung embeddings...


Batches: 100%|██████████| 4/4 [00:16<00:00,  4.01s/it]

Embeddings shape: (60, 384) 


In [10]:
# ════════════════════════════════════════════════════════════
# FUNGSI RETRIEVE
# ════════════════════════════════════════════════════════════
case_ids = df_f['case_id'].tolist()

def retrieve(query: str, k: int = 5, method: str = 'bert') -> List[Tuple[str, float]]:
    """
    Retrieve top-k kasus paling mirip.
    method: 'tfidf' atau 'bert'
    Returns: List[(case_id, similarity_score)]
    """
    if method == 'tfidf':
        q_vec = tfidf.transform([preprocess(query)])
        sims  = cosine_similarity(q_vec, X_all_tfidf).flatten()
    else:
        if bert_model is None:
            print('BERT tidak aktif, fallback ke TF-IDF')
            return retrieve(query, k, method='tfidf')
        q_emb = bert_model.encode(
            [query[:512]], normalize_embeddings=True, show_progress_bar=False
        )
        sims = (q_emb @ bert_embeddings.T).flatten()

    top_idx = np.argsort(sims)[::-1][:k]
    return [(case_ids[i], round(float(sims[i]), 4)) for i in top_idx]

# ── Test ───────────────────────────────────────────────────
q = 'terdakwa kedapatan membawa sabu seberat 5 gram saat razia polisi'
print('Query:', q)
print('\nTop-5 TF-IDF:')
for cid, sc in retrieve(q, k=5, method='tfidf'):
    row = df_f[df_f['case_id'] == cid].iloc[0]
    print(f'  {cid} | sim={sc:.4f} | vonis={row["vonis"]} | {row["no_perkara"]}')

if bert_model is not None:
    print('\nTop-5 BERT:')
    for cid, sc in retrieve(q, k=5, method='bert'):
        row = df_f[df_f['case_id'] == cid].iloc[0]
        print(f'  {cid} | sim={sc:.4f} | vonis={row["vonis"]} | {row["no_perkara"]}')

Query: terdakwa kedapatan membawa sabu seberat 5 gram saat razia polisi

Top-5 TF-IDF:
  case_029 | sim=0.0734 | vonis=pidana penjara selama 1 (satu) tahun dikurangi selama terdakwa beradadalam tahanan  | 1535/PID.B/2024/PN
  case_019 | sim=0.0731 | vonis=pidana penjara selama 8 (delapan) tahun dan 6(enam) bulan dikurangi masa penahanan  | 1294/PID.B/2025/PN
  case_003 | sim=0.0211 | vonis=nan | 1063/PID.B/2024/PN
  case_038 | sim=0.0190 | vonis=pidana penjara selama 1 (satu) tahun dan 2 (dua) bulan dikurangiselama terdakwa ber; pidana penjara selama 1(satu) tahun | 1904/PID.B/2024/PN
  case_051 | sim=0.0187 | vonis=pidana penjara selama 3 (tahun) tahun dan6 (enam) bulan dikurangi selama terdakwa b | 599/PID.B/2024/PN

Top-5 BERT:
  case_047 | sim=0.6627 | vonis=nan | 532/PID.B/2024/PN
  case_055 | sim=0.6190 | vonis=pidana penjara selama 1 (satu) tahun dan 6 (enam) bulan dikurangi selamaterdakwa be; tidak terbukti; bebas | 880/PID.B/2024/PN
  case_031 | sim=0.6026 | vonis=pidana penja

In [8]:
# ════════════════════════════════════════════════════════════
# BUAT QUERIES EVALUASI  →  data/eval/queries.json
# ════════════════════════════════════════════════════════════

# Query manual
manual_queries = [
    {'query_id':'Q001', 'query':'terdakwa memukul korban dengan tangan hingga luka memar di wajah',
    'expected_label':'ringan', 'description':'Penganiayaan ringan', 'true_case_id': None},
    {'query_id':'Q002', 'query':'terdakwa menyerang korban menggunakan senjata tajam parang menyebabkan luka sobek',
    'expected_label':'sedang', 'description':'Penganiayaan dengan senjata', 'true_case_id': None},
    {'query_id':'Q003', 'query':'terdakwa menganiaya korban secara berulang hingga korban meninggal dunia',
    'expected_label':'berat',  'description':'Penganiayaan berujung kematian', 'true_case_id': None},
    {'query_id':'Q004', 'query':'terdakwa melakukan kekerasan fisik terhadap anggota keluarga anak di bawah umur',
    'expected_label':'sedang', 'description':'KDRT terhadap anak', 'true_case_id': None},
    {'query_id':'Q005', 'query':'terdakwa berkelahi dengan korban di tempat umum mengakibatkan luka ringan',
    'expected_label':'ringan', 'description':'Perkelahian di tempat umum', 'true_case_id': None},
]

# Query dari test set aktual
real_queries = []
for i in idx_test[:10]:
    row = df_f.iloc[i]
    real_queries.append({
        'query_id'      : f'REAL_{row["case_id"]}',
        'query'         : (str(row['ringkasan_fakta']) + ' ' + str(row['pasal']))[:512],
        'expected_label': row['vonis'],
        'description'   : f'Kasus nyata {str(row["no_perkara"])[:40]}',
        'true_case_id'  : row['case_id'],
    })

all_queries = manual_queries + real_queries
with open(EVAL_DIR / 'queries.json', 'w', encoding='utf-8') as f:
    json.dump(all_queries, f, ensure_ascii=False, indent=2)

print(f' {len(all_queries)} query disimpan → data/eval/queries.json')
print(f'   Manual: {len(manual_queries)} | Real (test set): {len(real_queries)}')

 15 query disimpan → data/eval/queries.json
   Manual: 5 | Real (test set): 10
